# KRISS — Hybrid SMS Spam & Scam Detection Model Training
This Google Colab notebook follows the project SDS/SRS training design: **70/30 stratified split**, multilingual preprocessing, **TF-IDF unigram+bigram features**, comparison of **Naïve Bayes, Logistic Regression, SVM, and Random Forest**, 5-fold cross-validation, confusion matrix, Accuracy/Precision/Recall/F1 and ROC analysis. The deployed model predicts **Legitimate / Spam / Scam**.

> Dataset note: the bundled 10,000-row dataset is realistic **synthetic localized data** (English, Sinhala, Singlish), not private real SMS data. Replace or extend it with properly consented/licensed samples for final research validation.


In [ ]:
!pip -q install scikit-learn==1.5.2 pandas==2.2.3 joblib==1.4.2 matplotlib==3.9.2 seaborn==0.13.2


In [ ]:
import re, json, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, recall_score, f1_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize
warnings.filterwarnings('ignore')
RANDOM_STATE=42


## 1. Upload the dataset
Run this cell and upload `kriss_sms_multilingual_10000.csv` from the project `dataset/` folder.


In [ ]:
from google.colab import files
uploaded = files.upload()
DATASET_FILE = next(iter(uploaded.keys()))
print('Using:', DATASET_FILE)


**Alternative — Google Drive:** instead of the upload cell above, mount Drive and set `DATASET_FILE` to your Drive path.


In [ ]:
# OPTIONAL GOOGLE DRIVE METHOD
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_FILE = '/content/drive/MyDrive/KRISS/kriss_sms_multilingual_10000.csv'


In [ ]:
df=pd.read_csv(DATASET_FILE)
required={'message','label','language'}
assert required.issubset(df.columns), f'Missing columns: {required-set(df.columns)}'
df=df.dropna(subset=['message','label']).drop_duplicates(subset=['message']).copy()
df['message']=df['message'].astype(str).str.strip()
df=df[df['message'].str.len()>0]
valid={'Legitimate','Spam','Scam'}
assert set(df['label'].unique()).issubset(valid), df['label'].value_counts()
print('Rows:',len(df))
display(df.head())
print('\nClass distribution:')
display(df['label'].value_counts().to_frame())
print('\nLanguage distribution:')
display(df['language'].value_counts().to_frame())


In [ ]:
URL_RE=re.compile(r'https?://\S+|www\.\S+',re.I)
EMAIL_RE=re.compile(r'\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b',re.I)
PHONE_RE=re.compile(r'(?<!\d)(?:\+?94|0)?7\d{8}(?!\d)')
LONG_NUMBER_RE=re.compile(r'\b\d{4,}\b')
EN_STOP={"the","a","an","is","are","was","were","to","of","for","and","or","in","on","at","this","that","your","you","we","our","be","with"}
SI_STOP={"සහ","හා","මෙම","මේ","එක","වෙත","ඔබ","ඔබගේ","අපේ","අද","දින"}

def preprocess_text(text):
    text=unicodedata.normalize('NFKC',str(text)).lower().strip()
    text=URL_RE.sub(' urltoken ',text)
    text=EMAIL_RE.sub(' emailtoken ',text)
    text=PHONE_RE.sub(' phonetoken ',text)
    text=LONG_NUMBER_RE.sub(' numbertoken ',text)
    text=re.sub(r'[^\w\u0D80-\u0DFF\u0B80-\u0BFF]+',' ',text,flags=re.UNICODE)
    tokens=[t for t in text.split() if t not in EN_STOP and t not in SI_STOP and len(t)>1]
    return ' '.join(tokens)

df['clean_text']=df['message'].map(preprocess_text)
display(df[['message','clean_text','label','language']].sample(5,random_state=42))


### Why no aggressive stemming?
The SDS mentions stemming/lemmatization as a possible preprocessing step. For this **Sinhala + English + Singlish** dataset, aggressive English stemming can damage Sinhala/Singlish tokens. This notebook therefore uses Unicode normalization, conservative stop-word removal, anonymization tokens, and n-grams. If you later add a validated Sinhala stemmer/lemmatizer, compare it experimentally before deployment.


In [ ]:
X_train_text,X_test_text,y_train,y_test=train_test_split(
    df['clean_text'],df['label'],test_size=0.30,random_state=RANDOM_STATE,stratify=df['label'])
print('Training:',len(X_train_text),'Testing:',len(X_test_text))
print(pd.DataFrame({'train':y_train.value_counts(),'test':y_test.value_counts()}))


In [ ]:
vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=0.98,max_features=30000,sublinear_tf=True)
X_train=vectorizer.fit_transform(X_train_text)
X_test=vectorizer.transform(X_test_text)
print('TF-IDF shape:',X_train.shape)


In [ ]:
models={
 'Naive Bayes':MultinomialNB(),
 'Logistic Regression':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE),
 'SVM (calibrated)':CalibratedClassifierCV(estimator=LinearSVC(class_weight='balanced',random_state=RANDOM_STATE),cv=3),
 'Random Forest':RandomForestClassifier(n_estimators=100,class_weight='balanced_subsample',n_jobs=1,random_state=RANDOM_STATE)
}
param_grids={
 'Naive Bayes':{'alpha':[0.1,0.5,1.0]},
 'Logistic Regression':{'C':[0.5,1.0,2.0]},
 'SVM (calibrated)':{'estimator__C':[0.5,1.0,2.0]},
 'Random Forest':{'max_depth':[None,30],'min_samples_leaf':[1,2]}
}
cv5=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
results=[]; trained={}; best_params={}
for name,model in models.items():
    print('\nTuning',name)
    search=GridSearchCV(model,param_grids[name],cv=3,scoring='f1_macro',n_jobs=-1,refit=True)
    search.fit(X_train,y_train)
    tuned=search.best_estimator_
    best_params[name]=search.best_params_
    # SDS requires five-fold cross-validation; score the tuned estimator on training data.
    cv_scores=cross_val_score(tuned,X_train,y_train,cv=cv5,scoring='f1_macro',n_jobs=-1)
    trained[name]=tuned
    pred=tuned.predict(X_test)
    acc=accuracy_score(y_test,pred)
    macro_f1=f1_score(y_test,pred,average='macro')
    scam_recall=recall_score(y_test,pred,labels=['Scam'],average='macro',zero_division=0)
    results.append({
        'model':name,'cv_f1_mean':cv_scores.mean(),'cv_f1_std':cv_scores.std(),
        'accuracy':acc,'macro_f1':macro_f1,'scam_recall':scam_recall,
        'best_params':str(search.best_params_)
    })
results_df=pd.DataFrame(results).sort_values(['scam_recall','macro_f1','accuracy'],ascending=False)
display(results_df.style.format({c:'{:.4f}' for c in ['cv_f1_mean','cv_f1_std','accuracy','macro_f1','scam_recall']}))


In [ ]:
# Prefer models meeting the SRS >=95% accuracy target, then prioritize Scam recall and macro F1.
eligible=results_df[results_df['accuracy']>=0.95]
selection_pool=eligible if len(eligible) else results_df
selection_pool=selection_pool.copy()
selection_pool['selection_score']=0.50*selection_pool['scam_recall']+0.30*selection_pool['macro_f1']+0.20*selection_pool['accuracy']
best_name=selection_pool.sort_values('selection_score',ascending=False).iloc[0]['model']
best_model=trained[best_name]
print('Selected model:',best_name)


In [ ]:
best_pred=best_model.predict(X_test)
print(classification_report(y_test,best_pred,digits=4))
cm=confusion_matrix(y_test,best_pred,labels=['Legitimate','Spam','Scam'])
plt.figure(figsize=(7,5));sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['Legitimate','Spam','Scam'],yticklabels=['Legitimate','Spam','Scam']);plt.title(f'Confusion Matrix — {best_name}');plt.xlabel('Predicted');plt.ylabel('True');plt.show()


In [ ]:
# Multiclass One-vs-Rest ROC analysis
classes=['Legitimate','Spam','Scam']
y_bin=label_binarize(y_test,classes=classes)
probs=best_model.predict_proba(X_test)
class_index={c:i for i,c in enumerate(best_model.classes_)}
plt.figure(figsize=(7,6))
roc_auc={}
for i,c in enumerate(classes):
    p=probs[:,class_index[c]]
    fpr,tpr,_=roc_curve(y_bin[:,i],p); roc_auc[c]=auc(fpr,tpr)
    plt.plot(fpr,tpr,label=f'{c} AUC={roc_auc[c]:.3f}')
plt.plot([0,1],[0,1],'--');plt.xlabel('False Positive Rate');plt.ylabel('True Positive Rate');plt.title('One-vs-Rest ROC');plt.legend();plt.show()


In [ ]:
acc=accuracy_score(y_test,best_pred)
precision,recall,f1,_=precision_recall_fscore_support(y_test,best_pred,average='macro',zero_division=0)
scam_recall=recall_score(y_test,best_pred,labels=['Scam'],average='macro',zero_division=0)
metadata={
 'model_version':'KRISS-Colab-1.1', 'selected_model':best_name, 'best_params':best_params.get(best_name,{}), 'dataset_rows':int(len(df)),
 'train_rows':int(len(X_train_text)), 'test_rows':int(len(X_test_text)), 'split':'70/30 stratified',
 'classes':['Legitimate','Spam','Scam'], 'languages':sorted(df['language'].unique().tolist()),
 'accuracy':float(acc), 'macro_precision':float(precision), 'macro_recall':float(recall),
 'macro_f1':float(f1), 'scam_recall':float(scam_recall), 'roc_auc':{k:float(v) for k,v in roc_auc.items()},
 'preprocessing':'NFKC + lowercase + URL/email/phone/number tokens + conservative stopwords',
 'tfidf':{'ngram_range':[1,2],'min_df':2,'max_features':30000}
}
print(json.dumps(metadata,indent=2,ensure_ascii=False))


In [ ]:
# Performance check for the SRS <= 2 second classification target.
import time
sample_texts=X_test_text.iloc[:200].tolist()
start=time.perf_counter()
for text in sample_texts:
    x=vectorizer.transform([text])
    _=best_model.predict(x)
elapsed=time.perf_counter()-start
avg_ms=(elapsed/len(sample_texts))*1000
print(f'Average model inference (including vectorization): {avg_ms:.3f} ms/SMS over {len(sample_texts)} samples')
print('SRS <= 2 seconds target:', 'PASS' if avg_ms <= 2000 else 'REVIEW')
metadata['avg_inference_ms_colab']=float(avg_ms)


In [ ]:
OUT=Path('/content/kriss_model_export');OUT.mkdir(exist_ok=True)
joblib.dump(best_model,OUT/'sms_classifier.joblib')
joblib.dump(vectorizer,OUT/'tfidf_vectorizer.joblib')
(OUT/'model_metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print('Saved:',list(OUT.iterdir()))


In [ ]:
# Quick inference check before export
samples=[
 'Hi, are we still meeting at 3 PM today?',
 'FLASH SALE! Get 50% OFF this weekend. Reply JOIN.',
 'URGENT! Your bank account will be suspended. Verify password now at https://verify.example/account',
 'Suba pathum! oyata Rs. 100000 prize ekak win wela. danma claim karanna',
 'ඔබගේ පාර්සලය රඳවා ඇත. රු. 500 ගාස්තුව දැන් ගෙවන්න.'
]
for s in samples:
    x=vectorizer.transform([preprocess_text(s)])
    pred=best_model.predict(x)[0]; prob=best_model.predict_proba(x)[0]
    print('\n',s,'\n =>',pred, f'{max(prob)*100:.2f}%')


In [ ]:
!cd /content/kriss_model_export && zip -q /content/KRISS_Colab_Model.zip sms_classifier.joblib tfidf_vectorizer.joblib model_metadata.json
from google.colab import files
files.download('/content/KRISS_Colab_Model.zip')


## Deployment
Extract `KRISS_Colab_Model.zip` and copy the three files into `backend/model/`. Restart FastAPI. `GET /health` should then return `model_ready: true`.

For final academic evaluation, retain the printed tables/plots and report Accuracy, Precision, Recall, F1, Scam Recall, Confusion Matrix and ROC/AUC. The system design prioritizes avoiding **Scam → Legitimate false negatives**.
